In [4]:
##################################
#####testing all the metrics######
##################################

import cornac
from cornac.data import Reader
from cornac.datasets import movielens
from cornac.data import Dataset, FeatureModality
from cornac.eval_methods import RatioSplit, StratifiedSplit
from cornac.metrics import RMSE,AUC,NDCG,Precision,Recall
from cornac.models import MF, ItemKNN, UserKNN, NMF, BPR
import pandas as pd
import numpy as np
import random
import math
import seaborn as sns
import matplotlib.pyplot as plt

/Users/tahsinalamgirkheya/anaconda3/envs/cornac/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:

# movie_data = reader.read(fpath="./data/indexed_movies.csv", sep=",", fmt="UIRT")
# movie_data
movies = pd.read_csv("../cornac/data_c/indexed_movies.csv")

movies = movies.drop(columns=movies.columns[0])
movies[:4]

unique_genres = set("|".join(movies["genres"]).split("|"))
unique_genres = list(unique_genres)

for genre in unique_genres:
    movies[genre] = 0
for index, row in movies.iterrows():
    genres = row["genres"].split("|")
    for genre in genres:
        movies.at[index, genre] = 1

# item_categories = movies[unique_genres]
# unique_iids = rating_data_pd['itemID'].unique()
# movies = movies[movies['itemID'].isin(unique_iids)]
genre = movies[unique_genres]
item_features_numpy = genre.to_numpy()
# print(item_features_numpy.shape)

# item_categories = item_categor
item_features = {
    str(item_id): {"genre_" + str(idx): value for idx, value in enumerate(row)}
    for item_id, row in enumerate(item_features_numpy)
}
ids = list(range(0, 3416))
item_feature_modality = FeatureModality(
    features=item_features_numpy, ids=ids, normalized=True
)


users = pd.read_csv("../cornac/data_c/u_id_mapping.csv", sep="\t")
users = users.drop(columns=users.columns[0])
gender_map = {"M": 0, "F": 1}
users["Gender"] = users["Gender"].map(gender_map)
# unique_uids = rating_data_pd['userID'].unique()

# users = users[users["userID"].isin(unique_uids)]
user_features_numpy = users.to_numpy()
print(user_features_numpy.shape)
print(item_features_numpy.shape)
user_feature_modality = FeatureModality(
    features=user_features_numpy, name="user", normalized=True, ids=list(range(0, 6040))
)

# print("Example Item Features:")
# for item_id, features in list(item_features.items())[:5]:
#     print(f"Item ID: {item_id}, Features: {features}")

(6040, 2)
(3416, 18)


In [42]:
import numpy as np
from cornac.metrics import RatingMetric
import pandas as pd
import math


class GenreRPrecision(RatingMetric):
    def __init__(self, gender_df, unique_genres, **kwargs):
        """ 
        initializating genders of the users
        Parameters
        ----------
        gender_mapping : dict
            A dictionary mapping user IDs to their genders.
        """
        super().__init__(name="GenreRPrecision", **kwargs)
        self.gender_df = gender_df
        self.unique_genres = unique_genres

    def compute(self, reco_matrix, movies_genre_df, item_df):
        """
        reco_matrix : n_userxk np array containing the ranked recommended list for users
        item_df : pd df containing all items with ids and genre info as ohe
        movies_genre_df: pd df containing the total proportion for all genres for all movies
        returns the abs diff for each gender genre distribution
        """
        # precision of action = action movies / total action movies
        new_top_k = math.floor(movies_genre_df.max())
        df_reco = pd.DataFrame(
            {
                "userID": np.repeat(np.arange(reco_matrix.shape[0]), new_top_k),
                "itemID": reco_matrix.flatten(),
                "rank": np.tile(np.arange(1, new_top_k + 1), reco_matrix.shape[0]),
            }
        )
        merged_df = pd.merge(df_reco, item_df, on="itemID", how="inner")
        merged_df[self.unique_genres] = merged_df[self.unique_genres].div(
            merged_df[self.unique_genres].sum(axis=1), axis=0
        )
        
        for index, row in merged_df.iterrows():
            row_genres = row["genres"].split("|")
            for genre in row_genres:
                # print(genre in row_genres and row['rank'] > movies_df[genre])
                if row["rank"] > math.floor(movies_genre_df[genre]):
                    merged_df.at[index, genre] = 0
       
        reco_distribution = merged_df[["userID"] + self.unique_genres]
        reco_distribution = reco_distribution.groupby("userID")[
            self.unique_genres
        ].sum()
        print(reco_distribution)
        reco_distribution[self.unique_genres] = (
            reco_distribution[self.unique_genres] / movies_genre_df.values
        )
        print(reco_distribution)
        g_reco_distribution = self.get_gender_genre_dist(reco_distribution)
        return self.genre_result(g_reco_distribution)

    def get_gender_genre_dist(self, user_reco):
        """
        user_reco : is the recommended genre distibution for all users
        """
        recomen_df = pd.merge(user_reco, self.gender_df, on="userID")
        gender_genre_weights_r = recomen_df.groupby("Gender")[self.unique_genres].mean()
        distribution_gender = gender_genre_weights_r.sort_index()
        return distribution_gender

    def genre_result(self, gender_genre_dist):
        """
        gender_genre_dist : the genre distibution for each genre grouped by gender
        """
        gender_genre_dist = gender_genre_dist.to_numpy()
        return abs(gender_genre_dist[0] - gender_genre_dist[1])


In [11]:
reco_matrix_all_items=np.load('../reco_matrix_all_items.npy' )
reco_matrix_all_scores=np.load('../reco_matrix_all_scores.npy' )
itemknn=np.load('../itemknn.npy' )

#reco_matrix_all_items

In [8]:
user_ids = users.to_numpy()[:, 0]
item_ids = movies.to_numpy()[:, 2]
user_ids.__len__()

6040

In [21]:
# get the top_k ratings for all users:
top_k = 50
reco_matrix = np.zeros((len(user_ids), top_k), dtype=int)

for i in range(len(user_ids)):
    reco_matrix[i] = reco_matrix_all_items[user_ids[i]][:top_k]

In [15]:
movies_df = movies[unique_genres].div(movies[unique_genres].sum(axis=1), axis=0)
movies_df = movies_df[unique_genres].sum()
movies_df

Thriller       249.100000
Action         224.600000
War             64.433333
Crime           90.950000
Adventure      110.083333
Fantasy         22.383333
Western         42.166667
Documentary     79.500000
Mystery         45.950000
Horror         234.016667
Musical         52.850000
Drama          961.233333
Animation       42.116667
Romance        211.000000
Comedy         737.933333
Sci-Fi         121.283333
Film-Noir       22.083333
Children's     104.316667
dtype: float64

In [22]:
# for r-precision we wanna take the max of the proportiion of total genre in the movie dataset using ranks
movies_df = movies[unique_genres].div(movies[unique_genres].sum(axis=1), axis=0)
movies_df = movies_df[unique_genres].sum()
new_top_k = math.floor(movies_df.max())


# get the top_k ratings for all users:
top_k = 50
reco_matrix_r = np.zeros((len(user_ids), new_top_k), dtype=int)

for i in range(len(user_ids)):
    reco_matrix_r[i] = reco_matrix_all_items[user_ids[i]][:new_top_k]

# get the top_k ratings for all users:
# reco_matrix_top_r_2 = np.zeros((len(user_ids), new_top_k), dtype=int)

# for i in range(len(user_ids)):
#     reco_items, all_scores = models[1].rank(user_idx=user_ids[i], item_indices=list(item_ids))
#     reco_matrix_top_r_2[i] = reco_items[:new_top_k]


# grp = GenreRPrecision(users, unique_genres)
# MF_grp = grp.compute(reco_matrix_top_r, movies_df, movies)

In [43]:
gp = GenreRPrecision(users, unique_genres)
x=gp.compute(reco_matrix_r, movies_df, movies)


         Thriller     Action       War     Crime  Adventure   Fantasy  \
userID                                                                  
0       17.916667  16.833333  2.333333  0.500000   3.000000  0.000000   
1       11.666667  17.666667  0.833333  2.083333   1.700000  0.000000   
2       13.666667  16.000000  1.333333  2.833333   3.233333  0.000000   
3       14.583333  12.750000  0.833333  2.500000   1.200000  0.000000   
4       13.916667  10.750000  0.500000  1.333333   2.950000  0.000000   
...           ...        ...       ...       ...        ...       ...   
6035    16.000000  16.500000  1.166667  2.416667   1.616667  0.000000   
6036    15.250000  15.283333  0.500000  2.000000   1.866667  0.000000   
6037    15.833333  14.666667  0.500000  1.000000   4.000000  0.000000   
6038    16.250000  13.416667  1.000000  1.500000   6.166667  0.583333   
6039    14.416667  13.500000  1.000000  3.166667   3.700000  0.000000   

         Western  Documentary   Mystery     Horror